# Manga / Webtoon → 대사 추출 + 번역 (Colab)

파이프라인: `이미지/PDF → 대사 위치 찾기 → OCR → 언어 확인 → 외국어만 한국어 번역 → JSONL/TXT`

- 단일 이미지 / 여러 이미지 / PDF를 자동으로 구분합니다.
- 업로드 직후 썸네일로 입력을 확인합니다.
- 한국어는 번역하지 않고 그대로 출력합니다.
- 일본어는 기본적으로 MangaOCR, 한국어/중국어/영어는 PaddleOCR을 사용합니다.
- PaddleOCR은 CPU, RF-DETR과 Qwen 번역 모델은 Colab GPU(T4)를 사용합니다.
- 각 단계마다 진단 로그를 출력해서 문제가 생긴 위치를 찾기 쉽게 했습니다.


## 1. 패키지 설치

이 셀은 **처음 한 번만** 패키지를 설치합니다.

Paddle/PaddleOCR을 새로 설치한 직후에는 같은 Python 커널 안에 예전 모듈 상태가 남을 수 있어서, 설치가 끝나면 런타임을 자동으로 한 번 재시작합니다.
재시작 후 이 셀을 다시 실행하면 설치는 건너뜁니다.


In [ ]:
from pathlib import Path
import os
import signal
import subprocess
import sys

환경_설치_표시 = Path("/content/.manga2text_environment_ready")

if not 환경_설치_표시.exists():
    print("[설치 시작]")
    print("PaddleOCR은 CPU 버전으로 설치합니다.")
    print("Colab의 기존 PyTorch/CUDA 환경은 건드리지 않습니다.")

    삭제할_패키지 = [
        "paddlepaddle",
        "paddlepaddle-gpu",
        "paddleocr",
        "paddlex",
    ]

    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", *삭제할_패키지],
        check=False,
    )

    설치할_패키지 = [
        "paddlepaddle==3.2.2",
        "paddleocr==3.3.2",
        "paddlex==3.3.13",
        "rfdetr==1.7.0",
        "safetensors>=0.5",
        "huggingface_hub>=0.27",
        "manga-ocr>=0.1.11",
        "transformers>=4.51",
        "accelerate>=1.2",
        "bitsandbytes>=0.45",
        "lingua-language-detector>=2.0",
        "pymupdf>=1.24",
        "pillow",
        "numpy",
        "tqdm",
        "matplotlib",
    ]

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *설치할_패키지],
        check=True,
    )

    환경_설치_표시.write_text("ready", encoding="utf-8")

    print()
    print("[설치 완료]")
    print("PaddleX/PaddleOCR 중복 초기화를 막기 위해 런타임을 자동 재시작합니다.")
    print("재연결되면 이 셀부터 다시 실행하세요. 두 번째 실행부터는 설치를 건너뜁니다.")

    os.kill(os.getpid(), signal.SIGKILL)
else:
    print("[설치 확인]")
    print("필요한 패키지는 이미 설치되어 있습니다.")
    print("다음 셀로 진행하면 됩니다.")


## 2. 저장소 가져오기

파이프라인 함수가 들어 있는 GitHub 저장소를 현재 Colab 세션으로 가져옵니다.


In [ ]:
import shutil
import subprocess
from pathlib import Path

저장소_경로 = Path("/content/manga2text_tmp")

if 저장소_경로.exists():
    shutil.rmtree(저장소_경로)

subprocess.run(
    [
        "git",
        "clone",
        "-q",
        "https://github.com/HisameOgasahara/manga2text_tmp.git",
        str(저장소_경로),
    ],
    check=True,
)

print("[저장소 준비 완료]")
print(저장소_경로)


## 3. 실행 환경 확인

문제가 생기면 이 셀의 출력부터 확인하면 됩니다.

여기서는 PaddleX와 PaddleOCR을 직접 import하지 않고 설치된 버전 정보만 읽습니다.
이렇게 해서 PDX가 두 번 초기화되는 문제를 피합니다.


In [ ]:
import platform
from importlib.metadata import version

import paddle
import torch

print("[실행 환경]")
print("Python 버전            :", platform.python_version())
print("PyTorch 버전           :", torch.__version__)
print("PyTorch CUDA 사용 가능 :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름               :", torch.cuda.get_device_name(0))

print("Paddle 버전            :", paddle.__version__)
print("PaddleOCR 버전         :", version("paddleocr"))
print("PaddleX 버전           :", version("paddlex"))
print("Paddle CUDA 빌드       :", paddle.device.is_compiled_with_cuda())
print("Paddle 현재 장치       :", paddle.device.get_device())

if paddle.device.is_compiled_with_cuda():
    print("[주의] 이 노트북은 PaddleOCR을 CPU로 쓰도록 설계했습니다.")


## 4. 사용자 설정

아래 항목만 바꾸면 됩니다.

- **언어 자동 판별**: 앞부분 대사를 샘플로 읽어서 한국어/일본어/중국어/영어를 자동 판단합니다.
- **원문 언어**: 자동 판별을 끈 경우에만 사용합니다.
- **글자 읽는 방법**: `자동`이면 일본어는 MangaOCR, 나머지는 PaddleOCR을 사용합니다.
- **읽기 방향**: `자동`이면 일본어 만화는 오른쪽→왼쪽, 나머지는 왼쪽→오른쪽으로 처리합니다.


In [ ]:
import sys
from pathlib import Path

sys.path.append("/content/manga2text_tmp")

from manga2text_pipeline import (
    auto_detect_source_language,
    build_language_detector,
    classify_inputs,
    collect_page_images,
    describe_input_mode,
    load_koharu_detector,
    load_ocr_backend,
    load_translation_model,
    make_preview_images,
    process_pages,
    resolve_ocr_configuration,
    save_results,
)

# @title 사용자 설정

언어_자동_판별 = True  # @param {type:"boolean"}
원문_언어 = "한국어"  # @param ["한국어", "일본어", "중국어", "영어"]

글자_읽는_방법 = "자동"  # @param ["자동", "MangaOCR", "PaddleOCR"]
읽기_방향 = "자동"  # @param ["자동", "오른쪽→왼쪽 (일본 만화)", "왼쪽→오른쪽 (웹툰/영문)"]

외국어_한국어_번역 = True  # @param {type:"boolean"}
번역_모델 = "Qwen3-1.7B (가볍고 빠름)"  # @param ["Qwen3-1.7B (가볍고 빠름)", "Qwen3-4B (품질 우선)"]

효과음도_읽기 = False  # @param {type:"boolean"}
PDF_화질_DPI = 200  # @param {type:"integer"}
처리할_페이지_수 = 0  # @param {type:"integer"}
동시에_준비할_작업_수 = 4  # @param {type:"integer"}
진단_로그_보기 = True  # @param {type:"boolean"}

언어_코드 = {
    "한국어": "ko",
    "일본어": "ja",
    "중국어": "zh",
    "영어": "en",
}

OCR_코드 = {
    "자동": "auto",
    "MangaOCR": "manga",
    "PaddleOCR": "paddle",
}

읽기_방향_코드 = {
    "자동": "auto",
    "오른쪽→왼쪽 (일본 만화)": "rtl",
    "왼쪽→오른쪽 (웹툰/영문)": "ltr",
}

번역_모델_코드 = {
    "Qwen3-1.7B (가볍고 빠름)": "Qwen/Qwen3-1.7B",
    "Qwen3-4B (품질 우선)": "Qwen/Qwen3-4B",
}

AUTO_DETECT_SOURCE_LANGUAGE = 언어_자동_판별
SOURCE_LANGUAGE = 언어_코드[원문_언어]
OCR_BACKEND = OCR_코드[글자_읽는_방법]
READING_DIRECTION = 읽기_방향_코드[읽기_방향]
ENABLE_TRANSLATION = 외국어_한국어_번역
TRANSLATION_MODEL = 번역_모델_코드[번역_모델]
INCLUDE_SFX = 효과음도_읽기
PDF_DPI = PDF_화질_DPI
PAGE_LIMIT = None if 처리할_페이지_수 <= 0 else 처리할_페이지_수
INPUT_WORKERS = max(1, 동시에_준비할_작업_수)
DEBUG_LOG = 진단_로그_보기

# PaddleOCR은 CPU로 고정합니다.
PADDLE_DEVICE = "cpu"

MAX_NEW_TOKENS = 256
CROP_PADDING = 8
ROW_TOLERANCE = 80
AUTO_LANGUAGE_SAMPLE_CROPS = 3
DEBUG_SAMPLES_PER_PAGE = 3

CLASS_THRESHOLDS = {
    0: 0.25,  # 일반 텍스트
    1: 0.20,  # 효과음
    2: 0.50,  # 말풍선
    3: 0.50,  # 만화 칸
}

WORK_DIR = Path("/content/manga2text")
INPUT_DIR = WORK_DIR / "input"
PAGE_DIR = WORK_DIR / "pages"
OUTPUT_DIR = WORK_DIR / "output"

for directory in [INPUT_DIR, PAGE_DIR, OUTPUT_DIR]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print("[현재 설정]")
print("언어 자동 판별      :", AUTO_DETECT_SOURCE_LANGUAGE)
print("수동 원문 언어      :", 원문_언어)
print("글자 읽는 방법      :", 글자_읽는_방법)
print("읽기 방향           :", 읽기_방향)
print("PaddleOCR 계산 장치 : CPU")
print("외국어 번역         :", ENABLE_TRANSLATION)
print("번역 모델           :", 번역_모델)
print("효과음 읽기         :", INCLUDE_SFX)
print("PDF 화질            :", PDF_DPI, "DPI")
print("페이지 제한         :", PAGE_LIMIT if PAGE_LIMIT is not None else "전체")
print("동시 준비 작업 수   :", INPUT_WORKERS)
print("진단 로그           :", DEBUG_LOG)


## 5. 만화 업로드 + 자동 판별 + 썸네일 확인

이미지 1장, 여러 이미지, PDF, 이미지+PDF 혼합 입력을 모두 지원합니다.
이미지 여러 장은 파일명 순서로 처리합니다.


In [ ]:
import shutil

import matplotlib.pyplot as plt
from google.colab import files

if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)

INPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

uploaded_files = files.upload()

for filename, file_bytes in uploaded_files.items():
    destination = INPUT_DIR / filename
    destination.write_bytes(file_bytes)

input_groups = classify_inputs(INPUT_DIR)
input_mode = describe_input_mode(input_groups)

print("[입력 확인]")
print("입력 종류        :", input_mode)
print("이미지 수        :", len(input_groups["images"]))
print("PDF 수           :", len(input_groups["pdfs"]))
print("지원하지 않는 수 :", len(input_groups["unsupported"]))

for path in input_groups["unsupported"]:
    print("[경고] 지원하지 않는 파일:", path.name)

preview_items = make_preview_images(
    input_dir=INPUT_DIR,
    max_items=8,
    pdf_preview_pages=3,
)

if not preview_items:
    raise RuntimeError("미리보기 가능한 이미지 또는 PDF가 없습니다.")

column_count = min(4, len(preview_items))
row_count = (len(preview_items) + column_count - 1) // column_count

plt.figure(
    figsize=(4 * column_count, 5 * row_count),
)

for index, (label, image) in enumerate(preview_items, start=1):
    plt.subplot(row_count, column_count, index)
    plt.imshow(image)
    plt.title(label)
    plt.axis("off")

plt.tight_layout()
plt.show()


## 6. 페이지 이미지 준비

PDF 페이지 변환과 여러 이미지 준비는 CPU에서 병렬 처리합니다.


In [ ]:
if PAGE_DIR.exists():
    shutil.rmtree(PAGE_DIR)

PAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("[페이지 준비]")
print("동시 작업 수:", INPUT_WORKERS)
print("PDF DPI     :", PDF_DPI)
print("페이지 제한:", PAGE_LIMIT if PAGE_LIMIT is not None else "전체")

page_paths = collect_page_images(
    input_dir=INPUT_DIR,
    page_dir=PAGE_DIR,
    pdf_dpi=PDF_DPI,
    page_limit=PAGE_LIMIT,
    workers=INPUT_WORKERS,
)

print("준비된 페이지 수:", len(page_paths))

if not page_paths:
    raise RuntimeError("처리할 페이지가 없습니다.")


## 7. 대사 위치 찾기 모델(RF-DETR)

글자를 읽기 전에 페이지에서 대사와 텍스트가 있는 위치를 찾습니다.


In [ ]:
print("[대사 위치 모델 로드]")

detector = load_koharu_detector()

print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("사용 GPU:", torch.cuda.get_device_name(0))


## 8. 원문 언어와 OCR 방식 결정

자동 판별을 켜면 앞부분 텍스트를 여러 OCR 후보로 읽어 보고, 샘플 문자열과 점수를 출력합니다.


In [ ]:
if AUTO_DETECT_SOURCE_LANGUAGE:
    selected_source_language, auto_language_details = auto_detect_source_language(
        page_paths=page_paths,
        detector=detector,
        class_thresholds=CLASS_THRESHOLDS,
        crop_padding=CROP_PADDING,
        max_crops=AUTO_LANGUAGE_SAMPLE_CROPS,
        paddle_device=PADDLE_DEVICE,
    )
else:
    selected_source_language = SOURCE_LANGUAGE
    auto_language_details = None
    print("[언어 수동 선택]", 원문_언어)

ocr_config = resolve_ocr_configuration(
    source_language=selected_source_language,
    ocr_backend=OCR_BACKEND,
    reading_direction=READING_DIRECTION,
)

print("[결정된 처리 방법]")
print("원문 언어      :", ocr_config["source_language"])
print("사용할 OCR     :", ocr_config["ocr_backend"])
print("PaddleOCR 언어 :", ocr_config["paddle_lang"])
print("읽기 방향      :", ocr_config["reading_direction"])
print("PaddleOCR 장치 : CPU")


## 9. OCR 모델 로드


In [ ]:
ocr_model = load_ocr_backend(
    backend=ocr_config["ocr_backend"],
    paddle_lang=ocr_config["paddle_lang"],
    paddle_device=PADDLE_DEVICE,
)

print("[OCR 모델 로드 완료]")


## 10. 언어 확인 + 번역 모델 로드

한국어로 읽힌 대사는 번역하지 않습니다.


In [ ]:
language_detector, language_to_code = build_language_detector()

translation_tokenizer = None
translation_model = None

if ENABLE_TRANSLATION:
    translation_tokenizer, translation_model = load_translation_model(
        model_name=TRANSLATION_MODEL,
    )

    print("[번역 모델 로드 완료]")
    print(TRANSLATION_MODEL)
else:
    print("[번역 비활성화]")


## 11. 전체 파이프라인 실행

페이지별로 `대사 위치 찾기 → OCR → 언어 확인 → 필요하면 번역` 순서로 처리합니다.


In [ ]:
records = process_pages(
    page_paths=page_paths,
    detector=detector,
    ocr_backend=ocr_config["ocr_backend"],
    ocr_model=ocr_model,
    language_detector=language_detector,
    language_to_code=language_to_code,
    class_thresholds=CLASS_THRESHOLDS,
    reading_direction=ocr_config["reading_direction"],
    row_tolerance=ROW_TOLERANCE,
    crop_padding=CROP_PADDING,
    include_sfx=INCLUDE_SFX,
    enable_translation=ENABLE_TRANSLATION,
    translation_tokenizer=translation_tokenizer,
    translation_model=translation_model,
    max_new_tokens=MAX_NEW_TOKENS,
    debug=DEBUG_LOG,
    debug_samples_per_page=DEBUG_SAMPLES_PER_PAGE,
)

print("추출된 대사 수:", len(records))


## 12. 결과 미리보기


In [ ]:
for record in records[:30]:
    print(
        f"[p.{record['page']:03d} / {record['order']:02d}] "
        f"{record['language']} | "
        f"{record['original']} "
        f"-> {record['korean']}"
    )


## 13. JSONL / TXT 저장 + 다운로드


In [ ]:
from google.colab import files

jsonl_path, txt_path = save_results(
    records=records,
    output_dir=OUTPUT_DIR,
)

print("[저장 완료]")
print("JSONL:", jsonl_path)
print("TXT  :", txt_path)

files.download(str(jsonl_path))
files.download(str(txt_path))
